In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
import sgml, sgpp

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    sgpp.PolarsProcessor(predefined_types={'id': pl.Int64}),
    sgpp.ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [25]:
import importlib
import exp

importlib.reload(exp)

<module 'exp' from '/home/sun9sun9/jnote/sunkusun9/kaggle/PGS5/PGS5_ep11/exp.py'>

In [26]:
e = exp.Experimenter(df_train, sp = skf, sp_v = ss_v, y = y)

In [27]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [28]:
e.add_grp('preprocessor', method = 'transform')

In [29]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [30]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [31]:
e.set_node('lr1', 'lr')

In [32]:
e.get_data(1, edges= [('lr1', None)])

(<generator object Experimenter.get_data.<locals>.train_data_func at 0x7fe1915f7540>,
         lr1__loan_paid_back_0  lr1__loan_paid_back_1
 id                                                  
 4                    0.096614               0.903386
 6                    0.062517               0.937483
 10                   0.140021               0.859979
 13                   0.072554               0.927446
 15                   0.063363               0.936637
 ...                       ...                    ...
 593981               0.123077               0.876923
 593987               0.113943               0.886057
 593989               0.180164               0.819836
 593992               0.052873               0.947127
 593993               0.061400               0.938600
 
 [197998 rows x 2 columns])

In [33]:
from sklearn.preprocessing import StandardScaler
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], sparse_output = False)

In [35]:
import col

In [36]:
e.set_node('lr2', 'lr', edges = [('ohe', col.ohe_drop_first)])

In [355]:
e.nodes['lr2'].objs_[0][0][0].X_

['ohe__gender_Male',
 'ohe__gender_Other',
 'ohe__marital_status_Married',
 'ohe__marital_status_Single',
 'ohe__marital_status_Widowed',
 'ohe__education_level_High School',
 "ohe__education_level_Master's",
 'ohe__education_level_Other',
 'ohe__education_level_PhD',
 'ohe__employment_status_Retired',
 'ohe__employment_status_Self-employed',
 'ohe__employment_status_Student',
 'ohe__employment_status_Unemployed',
 'ohe__loan_purpose_Car',
 'ohe__loan_purpose_Debt consolidation',
 'ohe__loan_purpose_Education',
 'ohe__loan_purpose_Home',
 'ohe__loan_purpose_Medical',
 'ohe__loan_purpose_Other',
 'ohe__loan_purpose_Vacation',
 'std__annual_income',
 'std__debt_to_income_ratio',
 'std__credit_score',
 'std__loan_amount',
 'std__interest_rate',
 'std__grade_subgrade_no']

In [326]:
e.get_data(1, edges= [('lr2', None)])

(<generator object Experimenter.get_data.<locals>.train_data_func at 0x7fde10bf4940>,
         lr2__loan_paid_back_0  lr2__loan_paid_back_1
 id                                                  
 4                    0.034468               0.965532
 6                    0.020644               0.979356
 10                   0.052810               0.947190
 13                   0.024300               0.975700
 15                   0.020462               0.979538
 ...                       ...                    ...
 593981               0.001697               0.998303
 593987               0.046015               0.953985
 593989               0.061514               0.938486
 593992               0.017719               0.982281
 593993               0.020044               0.979956
 
 [197998 rows x 2 columns])

In [327]:
from IPython.display import Markdown
Markdown(
    e.to_mermaid(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

**Experimenter (n_splits=3), max_depth=2**

In [328]:
Markdown(
    e.node_to_mermaid('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [329]:
e.add_grp('dim_reduction', method = 'transform')

In [330]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], n_components=0.9)

In [331]:
Markdown(
    e.to_mermaid(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_dim_reduction["dim_reduction"]
        node_pca["pca"]
        style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
    grp_preprocessor --> grp_dim_reduction
```

**Experimenter (n_splits=3), max_depth=2**

In [332]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

In [333]:
def print_v(X, org_X):
    print(X, org_X)
    return [True for i in X]

In [334]:
e.set_node('lr3', 'lr', edges = [('ohe', print_v), ('pca', None)])

⚠️  Updating existing node 'lr3'
['ohe__gender_Male', 'ohe__gender_Other', 'ohe__marital_status_Married', 'ohe__marital_status_Single', 'ohe__marital_status_Widowed', 'ohe__education_level_High School', "ohe__education_level_Master's", 'ohe__education_level_Other', 'ohe__education_level_PhD', 'ohe__employment_status_Retired', 'ohe__employment_status_Self-employed', 'ohe__employment_status_Student', 'ohe__employment_status_Unemployed', 'ohe__loan_purpose_Car', 'ohe__loan_purpose_Debt consolidation', 'ohe__loan_purpose_Education', 'ohe__loan_purpose_Home', 'ohe__loan_purpose_Medical', 'ohe__loan_purpose_Other', 'ohe__loan_purpose_Vacation'] ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
['ohe__gender_Male', 'ohe__gender_Other', 'ohe__marital_status_Married', 'ohe__marital_status_Single', 'ohe__marital_status_Widowed', 'ohe__education_level_High School', "ohe__education_level_Master's", 'ohe__education_level_Other', 'ohe__education_level_PhD', 'ohe__e

In [256]:
dir(e.nodes['ohe'].objs_[0][0][0].transformer)

['__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_build_request_for_signature',
 '_check_X',
 '_check_feature_names',
 '_check_get_feature_name_combiner',
 '_check_infrequent_enabled',
 '_check_n_features',
 '_compute_n_features_outs',
 '_compute_transformed_categories',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_fit',
 '_fit_infrequent_category_mapping',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_tags',
 '_identify_infrequent',
 '_map_drop_idx_to_infrequent',
 '_map_infrequent_categories',
 '_more_tags

In [33]:
e.nodes['pca'].objs_

[[(<exp.TransformProcessor at 0x7fde92f4e900>, None)],
 [(<exp.TransformProcessor at 0x7fde92f4ed80>, None)],
 [(<exp.TransformProcessor at 0x7fde92f4cb30>, None)]]

In [116]:
Markdown(
    e.node_to_mermaid('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (4 path(s) found)**

In [117]:
Markdown(
    e.node_to_mermaid('lr3', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>drop</b></td><td align='left'>first</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (4 path(s) found)**

In [119]:
e.nodes['std'].objs_[0][0][0].X_

['annual_income',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'interest_rate',
 'grade_subgrade_no']

In [120]:
e.nodes['std'].objs_[0][0][0].output_vars

['std__annual_income',
 'std__debt_to_income_ratio',
 'std__credit_score',
 'std__loan_amount',
 'std__interest_rate',
 'std__grade_subgrade_no']

In [38]:
e.nodes['lr2'].objs_[0][0][0].output_vars

['lr2__loan_paid_back_0', 'lr2__loan_paid_back_1']

In [39]:
e.nodes['lr2'].objs_[0][0][0].obj.classes_

array([0, 1], dtype=int8)

In [123]:
e.nodes['lr3']._fit()

In [43]:
e.set_node('lr1', grp = 'lr', edges = [('std', None)])

⚠️  Updating existing node 'lr1'


In [44]:
e.set_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [])

🔄 Rebuilding 3 node(s) affected by group 'lr' update
  ├─ Rebuilding 'lr1' (priority: 1)...
⚠️  Updating existing node 'lr1'
  ├─ Rebuilding 'lr2' (priority: 1)...
⚠️  Updating existing node 'lr2'
  ├─ Rebuilding 'lr3' (priority: 1)...
⚠️  Updating existing node 'lr3'
✅ Rebuild complete!


In [45]:
e.nodes['lr1'].org_attr

{'processor': None,
 'edges': [('std', None)],
 'X': None,
 'y': None,
 'method': None,
 'args': {}}

In [46]:
e.grps['lr'].get_attrs()

{'edges': [(None, ['loan_paid_back'])],
 'processor': sklearn.linear_model._logistic.LogisticRegression,
 'X': None,
 'y': 'loan_paid_back',
 'method': 'predict_proba',
 'params': {}}

In [47]:
Markdown(
    e.node_to_mermaid('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [37]:
e.add_grp('cb', parent_grp='clf', verbose = 0)

In [38]:
e.set_node('cb1',  grp = 'cb', processor=cb.CatBoostClassifier, edges = [(None, X_num), (None, X_cat)], cat_features = X_cat)

In [39]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_.keys()

dict_keys(['learn', 'validation_0', 'validation_1'])

In [41]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_['validation_1']

{'Logloss': [0.545640176444159,
  0.4545261982249257,
  0.3924281187854096,
  0.3510341039589898,
  0.3239541458979721,
  0.3071992843448588,
  0.29297247132642146,
  0.2838785815975975,
  0.2771099077201327,
  0.27254567129132223,
  0.2689671499412666,
  0.266286967426198,
  0.264585342653263,
  0.2634487739071244,
  0.26211185311949264,
  0.2611627696903622,
  0.2605882355327742,
  0.2598808824722369,
  0.2595015516671475,
  0.2590405003027948,
  0.25866192642656133,
  0.2582968352471853,
  0.25811710138780536,
  0.25785466712083766,
  0.2577281867392856,
  0.2575090335045703,
  0.2573391486364953,
  0.2572420140775843,
  0.2570992005817121,
  0.2569662770578564,
  0.25690910931181005,
  0.2568485454991886,
  0.2567487577055978,
  0.2566949630692388,
  0.25660248731629237,
  0.2564986964480767,
  0.25648115293403195,
  0.25635068592384674,
  0.25632727259610344,
  0.25621647451366497,
  0.2562052180775544,
  0.25619209811122096,
  0.25615263517304376,
  0.2560644722374284,
  0.256025

In [50]:
Markdown(
    e.node_to_mermaid('cb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_cb1["cb1"]
        cb1_dummy[ ]
        style cb1_dummy fill:none,stroke:none
    end
    style node_cb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_cb1
```

**Path from Root to 'cb1' (1 path(s) found)**

In [34]:
Markdown(
    e.to_mermaid(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_dim_reduction["dim_reduction"]
        node_pca["pca"]
        style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_dim_reduction --> grp_clf
    grp_preprocessor --> grp_clf
    grp_preprocessor --> grp_dim_reduction
```

**Experimenter (n_splits=3), max_depth=2**

In [55]:
e.nodes['lr3'].objs_[0][0][0].X

['ohe__gender_Male',
 'ohe__gender_Other',
 'ohe__marital_status_Married',
 'ohe__marital_status_Single',
 'ohe__marital_status_Widowed',
 'ohe__education_level_High School',
 "ohe__education_level_Master's",
 'ohe__education_level_Other',
 'ohe__education_level_PhD',
 'ohe__employment_status_Retired',
 'ohe__employment_status_Self-employed',
 'ohe__employment_status_Student',
 'ohe__employment_status_Unemployed',
 'ohe__loan_purpose_Car',
 'ohe__loan_purpose_Debt consolidation',
 'ohe__loan_purpose_Education',
 'ohe__loan_purpose_Home',
 'ohe__loan_purpose_Medical',
 'ohe__loan_purpose_Other',
 'ohe__loan_purpose_Vacation',
 'pca__pca0',
 'pca__pca1',
 'pca__pca2',
 'pca__pca3',
 'pca__pca4']

In [56]:
e._find_descendants('std')

{'lr1', 'lr3', 'pca'}